# Maps

In [1]:
import heapq
import math

# --- Map 1: EuclidTown Data ---
EUCLIDTOWN_COORDS = {
    'A': (0, 0), 'B': (2, 1), 'C': (4, 0), 'D': (1, 3),
    'E': (3, 4), 'F': (5, 3), 'G': (6, 1)
}
EUCLIDTOWN_GRAPH = {
    'A': {'B': 2.2, 'D': 3.2},
    'B': {'A': 2.2, 'C': 2.2, 'E': 3.6},
    'C': {'B': 2.2, 'F': 3.6},
    'D': {'A': 3.2, 'E': 2.4},
    'E': {'B': 3.6, 'D': 2.4, 'F': 2.4},
    'F': {'C': 3.6, 'E': 2.4, 'G': 2.2},
    'G': {'F': 2.2}
}
EUCLIDTOWN_START = 'A'
EUCLIDTOWN_GOAL = 'G'

# --- Map 2: TrapVille Data ---
TRAPVILLE_COORDS = {
    'S': (0, 0), 'X': (2, 2), 'Y': (2, -2), 'Z': (4, 0), 'G': (6, 0)
}
TRAPVILLE_GRAPH = {
    'S': {'X': 2.9, 'Y': 2.9},
    'X': {'S': 2.9, 'Z': 8.0},
    'Y': {'S': 2.9, 'Z': 2.6},
    'Z': {'X': 8.0, 'Y': 2.6, 'G': 2.0},
    'G': {'Z': 2.0}
}
TRAPVILLE_START = 'S'
TRAPVILLE_GOAL = 'G'


# --- Required Utility & Heuristic Functions ---

def euclidean_heuristic(node, goal, coords):
    """Calculates the straight-line distance (Euclidean distance) to the goal as h(n)."""
    x1, y1 = coords[node]
    x2, y2 = coords[goal]
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def reconstruct_path(parents, goal):
    """Reconstructs the path from the start node to the goal node."""
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parents.get(current)
    return path[::-1] # Reverse the path

# Priority Queue Functions (using heapq directly for efficiency)
# The search functions will manage the tuple structure: (priority, state, g_cost, parent)
def priority_queue_push(pq, priority, state, g_cost, parent):
    """Pushes an item onto the min-heap."""
    # heapq will prioritize the tuple based on the first element (priority).
    # Secondary sorting (tie-breaking) is done by 'state' (node name) if priorities are equal.
    heapq.heappush(pq, (priority, state, g_cost, parent))

def priority_queue_pop(pq):
    """Pops the item with the smallest priority from the min-heap."""
    return heapq.heappop(pq)

#GBFS

In [2]:
# Assuming the Map Data and Utility Functions cell has been executed.

def gbfs(start, goal, graph, heuristic, coords):
    """
    Greedy Best-First Search (GBFS) implementation.
    [cite_start]The priority is determined solely by the heuristic cost, h(n)[cite: 9, 55].
    """
    # Min-heap: (priority=h(n), state, g_cost, parent)
    pq = []

    # Stores the parent of each node for path reconstruction
    parents = {start: None}

    # Nodes that have been expanded (visited)
    visited = set()
    visited_order = []
    expansions = 0

    # Initial state
    initial_h = heuristic(start, goal, coords)
    priority_queue_push(pq, initial_h, start, 0, None)

    while pq:
        expansions += 1

        # Pop the state with the lowest h(n)
        priority, current_state, current_g, parent_state = priority_queue_pop(pq)

        if current_state in visited:
            # Skip if already expanded
            continue

        # Log the expansion
        visited.add(current_state)
        visited_order.append(current_state)
        parents[current_state] = parent_state

        # Goal check upon expansion
        if current_state == goal:
            path = reconstruct_path(parents, goal)
            return path, current_g, expansions, visited_order

        # Explore neighbors
        for neighbor, edge_cost in graph.get(current_state, {}).items():
            if neighbor not in visited:
                new_g = current_g + edge_cost
                h_cost = heuristic(neighbor, goal, coords)

                # GBFS Priority: h(n)
                priority = h_cost

                priority_queue_push(pq, priority, neighbor, new_g, current_state)

    return [], 0, expansions, visited_order # Goal unreachable

A*

In [3]:
# Assuming the Map Data and Utility Functions cell has been executed.

def astar(start, goal, graph, heuristic, coords):
    """
    A* Search implementation.
    [cite_start]The priority is determined by f(n) = g(n) + h(n)[cite: 10, 57].
    [cite_start]It maintains the best g(n) found so far for a state (best-g)[cite: 57].
    """
    # Min-heap: (priority=f(n), state, g_cost, parent)
    pq = []

    # Store the best g_cost found for each state (best-g)
    g_costs = {start: 0}

    # Parent pointers for path reconstruction
    parents = {start: None}

    # Nodes that have been expanded (closed set)
    closed_set = set()
    visited_order = []
    expansions = 0

    # Initial state f(A) = g(A) + h(A) = 0 + h(A)
    initial_h = heuristic(start, goal, coords)
    initial_f = initial_h
    priority_queue_push(pq, initial_f, start, 0, None)

    while pq:
        expansions += 1

        # Pop the state with the lowest f(n)
        priority, current_state, current_g, parent_state = priority_queue_pop(pq)

        # Check if we found a better path earlier (best-g check is for the open set,
        # but this check against the closed set prevents re-expansion).
        if current_state in closed_set:
            continue

        # Log the expansion and add to closed set
        closed_set.add(current_state)
        visited_order.append(current_state)
        parents[current_state] = parent_state

        # Goal check upon expansion
        if current_state == goal:
            path = reconstruct_path(parents, goal)
            return path, current_g, expansions, visited_order

        # Explore neighbors
        for neighbor, edge_cost in graph.get(current_state, {}).items():
            # Calculate the cost of the path to the neighbor through the current state
            new_g = current_g + edge_cost

            # A* best-g check: Is the new path to 'neighbor' better than any previously found path?
            if new_g < g_costs.get(neighbor, float('inf')):

                g_costs[neighbor] = new_g # Update best-g
                h_cost = heuristic(neighbor, goal, coords)
                new_f = new_g + h_cost

                # A* Priority: f(n) = g(n) + h(n)
                priority_queue_push(pq, new_f, neighbor, new_g, current_state)

    return [], 0, expansions, visited_order # Goal unreachable